In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [ ]:
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА ДАННЫХ
# ======================
df = pd.read_csv(DATA / "store_sales.csv", parse_dates=["date"])
df = df.sort_values(["store_id", "date"]).reset_index(drop=True)



In [ ]:
print("Number of rows:", df.shape[0])
print("Columns:", list(df.columns))
print("Stores:", sorted(df["store_id"].unique()))
print("Date range:", df["date"].min().date(), "→", df["date"].max().date())



In [ ]:
df.info()
df.head()



In [ ]:
df.groupby("store_id")["sales"].describe()



In [ ]:
# Линии продаж по магазинам
plt.figure(figsize=(12, 5))
for store_id, g in df.groupby("store_id"):
    plt.plot(g["date"], g["sales"], label=f"store {store_id}", alpha=0.85)

plt.title("Daily sales by store", fontsize=16, pad=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Sales", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "store_sales_lines.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Средние продажи по магазинам (столбцы)
mean_sales = df.groupby("store_id")["sales"].mean().sort_index()

plt.figure(figsize=(7, 4))
sns.barplot(
    x=mean_sales.index.astype(str),
    y=mean_sales.values,
    hue=mean_sales.index.astype(str),
    palette="viridis",
    legend=False,
)
plt.title("Mean sales by store", fontsize=14, pad=12)
plt.xlabel("store_id")
plt.ylabel("Mean sales")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for i, v in enumerate(mean_sales.values):
    plt.text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()



In [ ]:
# Распределение продаж
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df["sales"], bins=30, color="#4c72b0", edgecolor="black")
plt.title("Sales histogram")
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="store_id", y="sales", palette="viridis")
plt.title("Sales by store (boxplot)")
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. LAG-ПРИЗНАКИ
# ======================
# Строим лаги внутри каждого магазина, чтобы не смешивать ряды
frames = []
for store_id, g in df.groupby("store_id"):
    g = g.copy()
    g["lag1"] = g["sales"].shift(1)
    g["lag7"] = g["sales"].shift(7)
    g["dow"] = g["date"].dt.dayofweek
    frames.append(g)

feat = pd.concat(frames, ignore_index=True).dropna()
print("Rows after lags:", feat.shape[0])
feat.head()



In [ ]:
# ======================
# 3. TRAIN / TEST ПО ВРЕМЕНИ
# ======================
# Берём один магазин для простого baseline (store 1)
g1 = feat[feat["store_id"] == 1].copy()
split_date = g1["date"].quantile(0.8)
train = g1[g1["date"] <= split_date]
test = g1[g1["date"] > split_date]
feats = ["lag1", "lag7", "dow"]
print("Train:", len(train), "Test:", len(test), "split:", split_date.date())



In [ ]:
# ======================
# 4. RIDGE РЕГРЕССИЯ
# ======================
model = Ridge(alpha=2.0)
model.fit(train[feats], train["sales"])
pred = model.predict(test[feats])

mae = mean_absolute_error(test["sales"], pred)
rmse = mean_squared_error(test["sales"], pred, squared=False)
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))



In [ ]:
# Коэффициенты модели
coef = pd.Series(model.coef_, index=feats)
print(coef)
print("intercept:", round(model.intercept_, 3))



In [ ]:
# Факт vs прогноз
plt.figure(figsize=(12, 4))
plt.plot(test["date"], test["sales"], label="actual", color="blue")
plt.plot(test["date"], pred, label="pred (Ridge)", color="red", alpha=0.8)
plt.title("Store 1 — forecast vs actual", fontsize=14, pad=12)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# Scatter факт/прогноз
plt.figure(figsize=(5, 5))
plt.scatter(test["sales"], pred, alpha=0.6, c="#4c72b0")
lims = [min(test["sales"].min(), pred.min()), max(test["sales"].max(), pred.max())]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# Ошибки по дням недели
err = test.copy()
err["ae"] = np.abs(err["sales"] - pred)

plt.figure(figsize=(8, 4))
sns.barplot(data=err, x="dow", y="ae", palette="viridis", errorbar=None)
plt.title("Mean absolute error by day of week", fontsize=14, pad=12)
plt.xlabel("Day of week (0=Mon)")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()



In [ ]:
# Мульти-магазинный Ridge (все магазины вместе)
train_all = feat[feat["date"] <= split_date]
test_all = feat[feat["date"] > split_date]
m_all = Ridge(2.0).fit(train_all[feats], train_all["sales"])
pred_all = m_all.predict(test_all[feats])
print("All stores MAE:", round(mean_absolute_error(test_all["sales"], pred_all), 3))

